In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier


spark = SparkSession.builder \
    .appName("Olist") \
    .getOrCreate()

In [2]:
!pip install kagglehub

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


In [4]:
#os.listdir(path)


tabelas = {}
for arquivo in os.listdir(path):
    if arquivo.endswith(".csv"):
        nome = arquivo.replace(".csv", "")

        tabelas[nome] = spark.read.csv(
            os.path.join(path, arquivo),
            header=True,
            inferSchema=True
        )

In [5]:
# Descrição das tabelas
descricoes = {
    "olist_customers_dataset": "dados dos clientes e localização do cliente",
    "olist_geolocation_dataset": "coordenadas geográficas associadas aos CEPs",
    "olist_order_items_dataset": "produtos vendidos em cada pedido",
    "olist_order_payments_dataset": "pagamentos dos pedidos",
    "olist_order_reviews_dataset": "Avaliações dos usuarios",
    "olist_orders_dataset": "status dos pedidos",
    "olist_products_dataset": "infos dos produtos",
    "olist_sellers_dataset": "dados dos vendedores",
    "product_category_name_translation": "Tradução das categorias"
}



for nome, df in tabelas.items():
    print(f"\n{'='*60}")
    print(f"TABELA: {nome}")
    print(f"Linhas: {df.count():,}")
    print(f"Colunas: {len(df.columns)}")
    print(f"Descrição: {descricoes.get(nome, 'Descrição não cadastrada.')}")
    print(f"Colunas: {', '.join(df.columns)}")


TABELA: olist_customers_dataset
Linhas: 99,441
Colunas: 5
Descrição: dados dos clientes e localização do cliente
Colunas: customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state

TABELA: olist_sellers_dataset
Linhas: 3,095
Colunas: 4
Descrição: dados dos vendedores
Colunas: seller_id, seller_zip_code_prefix, seller_city, seller_state

TABELA: olist_order_reviews_dataset
Linhas: 104,162
Colunas: 7
Descrição: Avaliações dos usuarios
Colunas: review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp

TABELA: olist_order_items_dataset
Linhas: 112,650
Colunas: 7
Descrição: produtos vendidos em cada pedido
Colunas: order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value

TABELA: olist_products_dataset
Linhas: 32,951
Colunas: 9
Descrição: infos dos produtos
Colunas: product_id, product_category_name, product_name_lenght, product_description_lenght, product_p

In [6]:


orders = spark.read.csv(
    os.path.join(path, "olist_orders_dataset.csv"),
    header=True,
    inferSchema=True
)

orders.show(5)
orders.printSchema()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [7]:
orders.count()

99441

In [8]:
orders.select("order_status").distinct().show()

+------------+
|order_status|
+------------+
|     shipped|
|    canceled|
|    approved|
|    invoiced|
|     created|
|   delivered|
| unavailable|
|  processing|
+------------+



In [9]:
produtos = tabelas["olist_products_dataset"]

produtos.show()

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [10]:
itens = tabelas["olist_order_items_dataset"]

itens.show()

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35|  58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13| 239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30| 199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51| 199.9|        18.14|
|00048cc3ae777c65d...|            1|ef92

In [11]:
#orders
#orders id
#product

orders = tabelas["olist_orders_dataset"]
items = tabelas["olist_order_items_dataset"]
products = tabelas["olist_products_dataset"]

df = (
    orders
    .join(items.select("order_id", "product_id"), "order_id", "inner")
    .join(products, "product_id", "left")
)

df.show()

+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------------------+-------------------+-------

In [12]:
#CRIACAO DO DF COM UNIAO DE ORDENS, DETALHES E PRODUTOS

df = (
    df
    .withColumn("order_purchase_timestamp",
                to_timestamp("order_purchase_timestamp"))
    .withColumn("order_delivered_customer_date",
                to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date",
                to_timestamp("order_estimated_delivery_date"))
)

df.show()

print("Linhas:", df.count())
print("Colunas:", len(df.columns))

+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+---------------------+-------------------+-------

In [13]:
# registro apenas dos pedidos com  data de chegada e status entregue

df = df.filter(
    (col("order_delivered_customer_date").isNotNull() & (col("order_status") == "delivered"))
)

In [14]:
#criacao de colunas para contagem de datas
df = df.withColumn("dias_realizado_envio",  datediff("order_delivered_customer_date", "order_purchase_timestamp"))\
.withColumn("dias_estimados_envio",datediff("order_estimated_delivery_date","order_purchase_timestamp"))\
.withColumn("dias_de_atraso",datediff("order_delivered_customer_date","order_estimated_delivery_date"))\
.withColumn("pedido_atrasado", when(col("dias_de_atraso")>0, 1).otherwise(0))\
.withColumn("volume_do_produto",col("product_length_cm") *col("product_height_cm") *col("product_width_cm"))\
.withColumn("mes_compra",month("order_purchase_timestamp"))\
.withColumn("dia_da_semana",dayofweek("order_purchase_timestamp"))\
.withColumn("hora_da_compra",hour("order_purchase_timestamp"))

#df.select("order_id", "pedido_atrasado").distinct().groupby('pedido_atrasado').count().show()

#df.show()


In [15]:
#Checar valores ausentes
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

#existem 1537 produtos sem categoria e 18 sem volume. Os 18 serao removidos da base e sera criada uma cateria "sem categorai"

#total = df.count()
#nulos = df.filter(col("product_category_name").isNull()).count()
#print(f"{nulos:,} de {total:,} ({nulos/total:.2%})")

df = df.dropna(
    subset=["product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"])


df = df.withColumn(
    "product_category_name",
    coalesce(col("product_category_name"), lit("sem_categoria"))
)

df = df.dropDuplicates()

+----------+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+--------------------+--------------------+--------------+---------------+-----------------+----------+-------------+--------------+
|product_id|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|dias_realizado_envio|dias_estimados_envio|dias_de_atraso|pedido_atrasado|volume_do_produto|mes_compra|dia_da_semana|hora_da_compra|
+----------+--------+-----------+------------+------------------

In [16]:
#SQL N1
#quais as categorias com maior numeros de dias de atraso
df.createOrReplaceTempView("df")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW pedidos_entregues AS
    SELECT *
    FROM df
""")


spark.sql("""
SELECT product_category_name, AVG(dias_de_atraso) FROM df
WHERE pedido_atrasado = 1
GROUP BY product_category_name
ORDER BY 2
""").show()

+---------------------+-------------------+
|product_category_name|avg(dias_de_atraso)|
+---------------------+-------------------+
| portateis_cozinha...|                3.0|
|            cine_foto|                3.0|
| construcao_ferram...|                3.0|
|                artes| 3.1666666666666665|
| fashion_roupa_fem...|                4.0|
|   artes_e_artesanato|                4.0|
| portateis_casa_fo...|               4.75|
|               flores|                5.0|
| construcao_ferram...|  5.666666666666667|
|    livros_importados|                6.0|
| construcao_ferram...|                6.0|
|      livros_tecnicos|  6.523809523809524|
|     eletrodomesticos|  6.696969696969697|
| livros_interesse_...|  6.878787878787879|
|         dvds_blu_ray|                7.0|
|             pet_shop|  7.919540229885057|
|      fashion_esporte|                8.0|
|        moveis_quarto|  8.333333333333334|
| fashion_underwear...|  8.363636363636363|
|                audio|  8.41463

In [17]:
#SQL N2
#quais as categorias com mais pedidos atrasados?
spark.sql("""
    SELECT product_category_name, COUNT(*)
    FROM df
    WHERE pedido_atrasado = 1
    GROUP BY product_category_name ORDER BY 2 DESC
""").show()

+---------------------+--------+
|product_category_name|count(1)|
+---------------------+--------+
|      cama_mesa_banho|     727|
|         beleza_saude|     657|
|        esporte_lazer|     497|
|     moveis_decoracao|     460|
| informatica_acess...|     425|
|   relogios_presentes|     412|
| utilidades_domest...|     311|
|            telefonia|     296|
|           automotivo|     280|
|           brinquedos|     246|
|   ferramentas_jardim|     230|
|                bebes|     226|
|           cool_stuff|     210|
|           perfumaria|     206|
|          eletronicos|     194|
|            papelaria|     153|
| fashion_bolsas_e_...|     105|
|    moveis_escritorio|     102|
|        sem_categoria|     101|
|             pet_shop|      87|
+---------------------+--------+
only showing top 20 rows


In [34]:
#SQL N3
#quais as categorais com maior taxa de atraso?
#df.groupBy("product_category_name").agg(avg("pedido_atrasado").alias("taxa_atraso")).orderBy(desc("taxa_atraso")).show(20)
spark.sql("""
SELECT product_category_name, AVG(pedido_atrasado)
FROM df
GROUP BY product_category_name
ORDER BY 2 DESC
""").show()

+---------------------+--------------------+
|product_category_name|avg(pedido_atrasado)|
+---------------------+--------------------+
| moveis_colchao_e_...| 0.13513513513513514|
|      casa_conforto_2|               0.125|
|                audio| 0.11781609195402298|
|        casa_conforto| 0.09476309226932668|
| fashion_underwear...| 0.09322033898305085|
|      livros_tecnicos| 0.08076923076923077|
|    moveis_escritorio|  0.0794392523364486|
|                bebes| 0.07915936952714536|
|     artigos_de_natal| 0.07874015748031496|
| portateis_cozinha...| 0.07692307692307693|
|          eletronicos| 0.07661927330173776|
| instrumentos_musi...|  0.0744336569579288|
|         beleza_saude| 0.07440543601359004|
|          moveis_sala| 0.07345971563981042|
|        sem_categoria| 0.07287157287157287|
|   relogios_presentes| 0.07268877911079746|
|      cama_mesa_banho| 0.07263462883404935|
|            alimentos| 0.07207207207207207|
|           automotivo| 0.07177646757241733|
| construc

In [19]:
#SQL N4
#em algum mes houve maior atraso?
spark.sql("""
    SELECT mes_compra, COUNT(*), AVG(pedido_atrasado)
    FROM df
    GROUP BY mes_compra ORDER BY 1 DESC
""").show()

+----------+--------+--------------------+
|mes_compra|count(1)|avg(pedido_atrasado)|
+----------+--------+--------------------+
|        12|    5712| 0.07352941176470588|
|        11|    7615|  0.1221273801707157|
|        10|    4957| 0.03853136978010894|
|         9|    4300| 0.04395348837209302|
|         8|   10989| 0.04768404768404769|
|         7|   10409|0.030646555865116727|
|         6|    9583|0.017426693102368777|
|         5|   10695| 0.05329593267882188|
|         4|    9462| 0.04903825829634327|
|         3|    9860|  0.1488843813387424|
|         2|    8444| 0.11712458550450024|
|         1|    8146|0.053277682298060396|
+----------+--------+--------------------+



In [20]:
#SQL N5
# quais as caracteristicas medias dos produtos que chegaram no prazo e atrasaram?
spark.sql("""
    SELECT pedido_atrasado, AVG(product_weight_g), AVG(volume_do_produto), AVG(dias_realizado_envio)
    FROM df
    GROUP BY pedido_atrasado ORDER BY 2 DESC
""").show()

+---------------+---------------------+----------------------+-------------------------+
|pedido_atrasado|avg(product_weight_g)|avg(volume_do_produto)|avg(dias_realizado_envio)|
+---------------+---------------------+----------------------+-------------------------+
|              1|    2455.066916729182|    17004.167141785445|        33.92513128282071|
|              0|   2059.7689477793106|    14951.867218497011|       10.903033997454736|
+---------------+---------------------+----------------------+-------------------------+



In [21]:
df_modelo = df.select(
    "pedido_atrasado",
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "volume_do_produto",
    "mes_compra",
    "dia_da_semana",
    "hora_da_compra")


features_numericas = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "volume_do_produto",
    "mes_compra",
    "dia_da_semana",
    "hora_da_compra"
]

df_modelo = df_modelo.dropna(subset=features_numericas)

treino, teste = df_modelo.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Treino:", treino.count())
print("Teste:", teste.count())

Treino: 79110
Teste: 19676


In [22]:
indexer = StringIndexer(
    inputCol="product_category_name",
    outputCol="categoria_index",
    handleInvalid="keep"
)

indexer_model = indexer.fit(treino)

treino = indexer_model.transform(treino)
teste = indexer_model.transform(teste)

In [23]:
treino.show()

+---------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+----------+-------------+--------------+---------------+
|pedido_atrasado|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|volume_do_produto|mes_compra|dia_da_semana|hora_da_compra|categoria_index|
+---------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+----------+-------------+--------------+---------------+
|              0| agro_industria_e_...|                 28|                       388|                 2|            3000|            18000|         1|            3|            12|           44.0|
|              0| agro_industria_e_...|                 28|                       388|                 2|            3000|            18000|         2|            1|            12|           44.0|
|              

In [24]:
encoder = OneHotEncoder(
    inputCols=["categoria_index"],
    outputCols=["categoria_vector"]
)

encoder_model = encoder.fit(treino)

treino = encoder_model.transform(treino)
teste = encoder_model.transform(teste)

In [25]:
assembler = VectorAssembler(
    inputCols=[
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "volume_do_produto",
        "mes_compra",
        "dia_da_semana",
        "hora_da_compra",
        "categoria_vector"
    ],
    outputCol="features"
)

treino = assembler.transform(treino)
teste = assembler.transform(teste)

In [26]:
# REGRESSAO LOGISTICA
regressao_logistica = LogisticRegression(
    featuresCol="features",
    labelCol="pedido_atrasado"
)

modelo_regressao_logistica = regressao_logistica.fit(treino)

previsoes = modelo_regressao_logistica.transform(teste)


tamanho = previsoes.count()

acuracia = previsoes.filter(
    previsoes.pedido_atrasado == previsoes.prediction).count() / tamanho

print("Acurácia:", acuracia)


Acurácia: 0.9333197804431795


In [27]:
#Arvore de decisao
from pyspark.ml.classification import DecisionTreeClassifier

arvore = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="pedido_atrasado",
    maxDepth=5
)

modelo_arvore = arvore.fit(treino)

previsoes_dt = modelo_arvore.transform(teste)

acuracia_dt = previsoes_dt.filter(
    previsoes_dt.pedido_atrasado == previsoes_dt.prediction
).count() / previsoes_dt.count()

print("Acurácia da Árvore de Decisão:", acuracia_dt)

Acurácia da Árvore de Decisão: 0.9331164870908721


In [28]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator



dados_cluster = df_modelo.select(
    "product_weight_g",
    "volume_do_produto",
    "product_photos_qty",
    "product_name_lenght",
    "product_description_lenght"
).dropna()

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=[
        "product_weight_g",
        "volume_do_produto",
        "product_photos_qty",
        "product_name_lenght",
        "product_description_lenght"
    ],
    outputCol="features"
)

dados_cluster = assembler.transform(dados_cluster)



In [29]:
avaliador = ClusteringEvaluator()

for k in [3, 4, 5, 6]:
    modelo = KMeans(k=k, seed=42).fit(dados_cluster)
    resultado = modelo.transform(dados_cluster)
    score = avaliador.evaluate(resultado)

    print("K =", k, "| Silhouette =", score)

K = 3 | Silhouette = 0.8582762988699427
K = 4 | Silhouette = 0.8454628216095079
K = 5 | Silhouette = 0.8132414610207238
K = 6 | Silhouette = 0.7914906524051023


In [30]:
kmeans = KMeans(
    k=3,
    seed=42
)

modelo_kmeans = kmeans.fit(dados_cluster)

resultado = modelo_kmeans.transform(dados_cluster)

resultado.groupBy("prediction").count().show()

+----------+-----+
|prediction|count|
+----------+-----+
|         1|81619|
|         2|14956|
|         0| 2211|
+----------+-----+



In [31]:
resultado.groupBy("prediction").avg(
    "product_weight_g",
    "volume_do_produto",
    "product_photos_qty",
    "product_name_lenght",
    "product_description_lenght"
).show()

+----------+---------------------+----------------------+-----------------------+------------------------+-------------------------------+
|prediction|avg(product_weight_g)|avg(volume_do_produto)|avg(product_photos_qty)|avg(product_name_lenght)|avg(product_description_lenght)|
+----------+---------------------+----------------------+-----------------------+------------------------+-------------------------------+
|         1|    1023.337592962423|     7107.577010254965|     2.2237469216726495|       48.74380965216432|              790.3805854029087|
|         2|    5804.244316662209|     42847.82227868414|      2.321208879379513|       49.89810109654988|              739.0772265311581|
|         0|   16396.145183175035|    123785.98959746721|     2.3251922207146087|      48.147896879240164|              965.7060153776572|
+----------+---------------------+----------------------+-----------------------+------------------------+-------------------------------+

